## 1. Análisis del rendimiento de newsletters

**Objetivo:** entender qué ha funcionado hasta ahora en las campañas enviadas desde Brevo y convertirlo en aprendizaje accionable.

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [ ]:
# Miramos el archivo

with open("data.csv", "r", encoding="utf-8", errors="replace") as f:
    for i in range(1, 10):
        print(i, f.readline().rstrip("\n"))
        
# Hay ';' y comillas raras

In [ ]:
#Expected 2 fields in line 4, saw 27
#significa que pandas está interpretando que el archivo tiene 2 columnas (por las primeras líneas), pero en la línea 4 aparece una fila con muchísimos separadores (o una línea “rota” por comillas / saltos de línea dentro de una celda).

# identificamos qué hay en las líneas problemáticas

from itertools import islice

with open("data.csv", "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(islice(f, 0, 10), start=1):
        print(f"{i:02d} | {line.rstrip()}")

In [54]:
df = pd.read_csv("data.csv", sep=";", engine="python", skiprows=3)

df.head()

,Campaign ID,Campaign Name,Name from,Email from,Sending date,Subject,Sent,Non delivered,Hard bounces,Soft bounces,Non delivered rate,Delivered,Total opens,Opens,Trackable open rate,Apple MPP Opens,Total clicked,Clicked,Click rate,Click-to-Open rate,Unsubscribed,Unsubscription rate,Delivered rate,Hard Bounces rate,Soft Bounces rate,Complaints,Complaints rate
0,5,Resumen Webinar enGira!,enGira!,info@engira.art,04-12-2024,Resumen Webinar enGira!,103,0,0,0,0%,103,247,87,"84,47%",19,34,19,"18,45%","27,94%",0,0%,100%,0%,0%,0,0%
1,6,Artistas Catálogo,enGira!,info@engira.art,23-10-2024,Participación Piloto enGira!,3,0,0,0,0%,3,13,3,100%,0,3,1,"33,33%","33,33%",0,0%,100%,0%,0%,0,0%
2,8,Artistas seleccionados,enGira!,info@engira.art,29-10-2024,Artistas seleccionados,38,2,0,2,"5,26%",36,773,44,100%,9,99,32,"88,89%","91,43%",0,0%,"94,74%",0%,"5,26%",0,0%
3,10,ARTISTAS TOOLKIT,enGira!,info@engira.art,28-10-2024,Participación Piloto enGira​!,122,5,2,3,"4,10%",117,974,134,100%,33,199,70,"59,83%","69,31%",0,0%,"95,90%","1,64%","2,46%",1,"0,82%"
4,12,Recordatorio,enGira!,info@engira.art,29-10-2024,Participación Piloto enGira!,24,1,0,1,"4,17%",23,214,27,100%,5,25,12,"52,17%","54,55%",0,0%,"95,83%",0%,"4,17%",0,0%


In [ ]:
df.tail()

In [ ]:
df.sample()

In [ ]:
df.info()

In [55]:
df.columns

Index(['Campaign ID', 'Campaign Name', 'Name from', 'Email from',
       'Sending date', 'Subject', 'Sent', 'Non delivered', 'Hard bounces',
       'Soft bounces', 'Non delivered rate', 'Delivered', 'Total opens',
       'Opens', 'Trackable open rate', 'Apple MPP Opens', 'Total clicked',
       'Clicked', 'Click rate', 'Click-to-Open rate', 'Unsubscribed',
       'Unsubscription rate', 'Delivered rate', 'Hard Bounces rate',
       'Soft Bounces rate', 'Complaints', 'Complaints rate'],
      dtype='str')

In [56]:
df = df.drop(columns=[
    "Name from", "Email from", "Hard bounces", 
    "Soft bounces", "Non delivered", "Delivered", 
    "Unsubscribed", "Complaints"])

In [57]:
df.columns

Index(['Campaign ID', 'Campaign Name', 'Sending date', 'Subject', 'Sent',
       'Non delivered rate', 'Total opens', 'Opens', 'Trackable open rate',
       'Apple MPP Opens', 'Total clicked', 'Clicked', 'Click rate',
       'Click-to-Open rate', 'Unsubscription rate', 'Delivered rate',
       'Hard Bounces rate', 'Soft Bounces rate', 'Complaints rate'],
      dtype='str')

In [58]:
import pandas as pd
import re

def limpiar_columnas(df):
    df = df.copy()
    
    df.columns = (
        df.columns
        .str.strip()                    # quitar espacios al inicio/fin
        .str.lower()                    # minúsculas
        .str.replace(r"[áàäâ]", "a", regex=True)
        .str.replace(r"[éèëê]", "e", regex=True)
        .str.replace(r"[íìïî]", "i", regex=True)
        .str.replace(r"[óòöô]", "o", regex=True)
        .str.replace(r"[úùüû]", "u", regex=True)
        .str.replace(r"ñ", "n", regex=True)
        .str.replace(r"[^\w\s]", "", regex=True)  # quitar símbolos (% , . etc)
        .str.replace(r"\s+", "_", regex=True)     # espacios → _
    )
    
    return df

In [60]:
df = limpiar_columnas(df)

In [63]:
df['sending_date'] = pd.to_datetime(df["sending_date"], errors="coerce")

In [65]:
df['sending_date'].dtype

dtype('<M8[us]')

In [66]:
df['year'] = df['sending_date'].dt.year
df['month'] = df['sending_date'].dt.month
df['day'] = df['sending_date'].dt.day
df['weekday'] = df['sending_date'].dt.weekday

In [85]:
df['subject'].value_counts()

subject
nuevas convocatorias                                            7
programa piloto engira!                                         4
demo engira!                                                    4
participación piloto engira!                                    2
fin del programa piloto engira!​                                2
descubre nuevos artistas escénicos                              2
presentes en dferia                                             2
presentamos engira! en madferia                                 2
resumen webinar engira!                                         1
artistas seleccionados                                          1
participación piloto engira​!                                   1
jornadas engira!                                                1
no te pierdas estas convocatorias!                              1
webinar engira! 3 de diciembre                                  1
jornadas de internacionalización y movilidad de artistas!       1
ar

In [84]:
df['subject'] = df['subject'].str.lower().str.strip()

In [80]:
def clasificar_campaña(nombre):
    
    if "convocatorias" in nombre:
        return "convocatorias"
    
    elif "jornadas" in nombre:
        return "jornadas"
    
    elif "demo" in nombre or 'webinar' in nombre or 'registro' in nombre:
        return "formacion"
    
    elif "dferia" in nombre or 'madferia' in nombre or 'mercartes' in nombre:
        return "ferias"
    
    elif "descubre" in nombre or "descobreix" in nombre or 'funcionamiento' in nombre or 'piloto' in nombre or 'destacados' in nombre:
        return "promocion"
    
    else:
        return "otros"

In [86]:
df['tipo_campaña'] = df['subject'].apply(clasificar_campaña)

In [87]:
df['tipo_campaña'].value_counts()


tipo_campaña
promocion        15
convocatorias    15
formacion        11
ferias            7
otros             3
jornadas          3
Name: count, dtype: int64

In [ ]:
df['campaign_name'] = df['campaign_name'].str.lower().str.strip()

In [79]:
df['campaign_name']

0                               resumen webinar engira!
1                                     artistas catálogo
2                                artistas seleccionados
3                                      artistas toolkit
4                                          recordatorio
5               invitación piloto engira! programadores
6     invitación piloto programadores madrid/galicia...
7     invitación piloto porgramadores extremadura-ca...
8              invitación piloto programadores cataluña
9                                 confirmación jornadas
10                                        convocatorias
11                                     webinar 3 de dic
12    jornadas de internacionalización y movilidad d...
13                                    artista destacado
14    jornadas de internacionalización y movilidad d...
15              invitación piloto engira! programadores
16                                 recordatorio webinar
17                                 nuevas convoc

In [92]:
def clasificar_publico(nombre):
    
    if 'artistas' in nombre or 'convocatorias' in nombre or 'webinar' in nombre or 'diagnostico'in nombre:
        
        return 'artistas'
    
    elif 'programadores' in nombre or 'demo' in nombre:
        
        return 'programadores'
    
    else:
        return 'all'

In [93]:
df['tipo_publico'] = df['campaign_name'].apply(clasificar_publico)

In [94]:
df['tipo_publico'].value_counts()

tipo_publico
artistas         27
programadores    16
all              11
Name: count, dtype: int64

In [90]:
df.columns

Index(['campaign_id', 'campaign_name', 'sending_date', 'subject', 'sent',
       'non_delivered_rate', 'total_opens', 'opens', 'trackable_open_rate',
       'apple_mpp_opens', 'total_clicked', 'clicked', 'click_rate',
       'clicktoopen_rate', 'unsubscription_rate', 'delivered_rate',
       'hard_bounces_rate', 'soft_bounces_rate', 'complaints_rate', 'year',
       'month', 'day', 'weekday', 'tipo_campaña', 'tipo_publico'],
      dtype='str')

- normalizar nombres columnas
- transformar fecha date_to_time y separar mes, año y día
- crear nuevas columnas: 
    - tipo de público
        - artistas
        - programadores
    - tipo de campaña
        - convocatorias (keywords: convocatorias)
        - jornadas
        - ferias (keywords: dferia, madferia, mercartes, feria)
        - promocion (keywords: descubre, descobreix, funcionamiento,piloto, destacados)
        - formacion (keywords: webinar, demo, registro, catalogo, seleccionados, toolkit)

- eliminar columnas:
    - name_from
    - Email_from
    - Hard Bounces
    - Soft bounces
    - Non delivered
    - delivered
    - unsusbscribed
    - complaints
